# FraudWatch

## Import packages

In [ ]:
import joblib
import pandas as pd
import streamlit as st
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

## Load and combine datasets

In [ ]:
# Filepaths
train_path = "fraudTrain.csv"
test_path = "fraudTest.csv"

# Load the files
df_train = pd.read_csv(train_path, index_col=0)
df_test = pd.read_csv(test_path, index_col=0)

# Combine into single dataframe
df = pd.concat([df_train, df_test], axis=0, ignore_index=True)

# The full dataset contains over 1.8 million records, which would slow down training and
# testing greatly to process in full. For the sake of our assignment, we will be only
# be taking the first 100000 rows of the test dataset.

# Take a sample of 50000 rows
df = df.sample(50000)
df.head()

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
1541144,2020-09-18 07:13:39,5359543825610251,"fraud_Jenkins, Hauck and Friesen",gas_transport,59.91,Michael,Francis,M,1833 Jeanette Stravenue,Belgrade,...,45.7801,-111.1439,18182,"Engineer, drilling",1975-06-29,784eb609215cbaf3725642a7a9f8bb57,1379488419,45.274075,-111.649432,0
1731581,2020-12-05 17:48:25,5540636818935089,fraud_Jast-McDermott,shopping_pos,3.96,Kenneth,Foster,M,329 Michael Extension,Lawrence,...,42.6911,-71.1605,76383,Geoscientist,1985-04-04,74e0ceb1e273a4a37f56dd2b5e6ce03d,1386265705,43.356278,-71.008959,0
354659,2019-06-15 11:24:44,2720894374956739,fraud_Bartoletti-Wunsch,gas_transport,51.17,Audrey,Hickman,F,3325 Gregory Square,Mount Clemens,...,42.5978,-82.8823,16305,"Psychologist, sport and exercise",1927-05-25,93c2b2ce1e96db464d9a7c7f2e9fb1dd,1339759484,42.372483,-83.508020,0
1493788,2020-08-29 22:50:25,6011438889172900,"fraud_Roob, Conn and Tremblay",shopping_pos,2.06,Allison,Allen,F,40624 Rebecca Spurs,De Witt,...,34.2853,-91.3336,5161,Electrical engineer,1993-04-08,a269c6b9d9a1603bee720d3ad4fdcfa8,1377816625,33.833389,-91.158293,0
468148,2019-07-25 15:50:35,60495593109,"fraud_Kilback, Nitzsche and Leffler",travel,6.58,Randall,Dillon,M,4440 George Mills Suite 591,Dallas,...,32.7699,-96.7430,1263321,Television camera operator,1942-11-24,9ce4d94b27e36cc2c79bb02351831722,1343231435,32.458643,-96.577001,0


## Data Cleaning


In [ ]:
df["merchant"] = df["merchant"].apply(lambda x: str(x).replace("fraud_", ""))

In [ ]:
# Drop unused text/unique string identifiers to prevent data leakage
drop_cols = ["trans_num", "first", "last", "street", "city", "state", "zip", "dob"]
df_clean = df.drop(columns=[col for col in drop_cols if col in df.columns])

# Convert timestamp to temporal features
if "trans_date_trans_time" in df_clean.columns:
    df_clean["trans_date_trans_time"] = pd.to_datetime(
        df_clean["trans_date_trans_time"]
    )
    df_clean["hour"] = df_clean["trans_date_trans_time"].dt.hour
    df_clean["day_of_week"] = df_clean["trans_date_trans_time"].dt.dayofweek
    df_clean.drop(columns=["trans_date_trans_time"], inplace=True)

In [ ]:
# Encode categorical variables like job or gender
label_encoders = {}
categorical_cols = ["merchant", "category", "gender", "job"]

for col in categorical_cols:
    if col in df_clean.columns:
        le = LabelEncoder()
        df_clean[col] = le.fit_transform(df_clean[col].astype(str))
        label_encoders[col] = le

df.head()

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
1541144,2020-09-18 07:13:39,5359543825610251,"Jenkins, Hauck and Friesen",gas_transport,59.91,Michael,Francis,M,1833 Jeanette Stravenue,Belgrade,...,45.7801,-111.1439,18182,"Engineer, drilling",1975-06-29,784eb609215cbaf3725642a7a9f8bb57,1379488419,45.274075,-111.649432,0
1731581,2020-12-05 17:48:25,5540636818935089,Jast-McDermott,shopping_pos,3.96,Kenneth,Foster,M,329 Michael Extension,Lawrence,...,42.6911,-71.1605,76383,Geoscientist,1985-04-04,74e0ceb1e273a4a37f56dd2b5e6ce03d,1386265705,43.356278,-71.008959,0
354659,2019-06-15 11:24:44,2720894374956739,Bartoletti-Wunsch,gas_transport,51.17,Audrey,Hickman,F,3325 Gregory Square,Mount Clemens,...,42.5978,-82.8823,16305,"Psychologist, sport and exercise",1927-05-25,93c2b2ce1e96db464d9a7c7f2e9fb1dd,1339759484,42.372483,-83.508020,0
1493788,2020-08-29 22:50:25,6011438889172900,"Roob, Conn and Tremblay",shopping_pos,2.06,Allison,Allen,F,40624 Rebecca Spurs,De Witt,...,34.2853,-91.3336,5161,Electrical engineer,1993-04-08,a269c6b9d9a1603bee720d3ad4fdcfa8,1377816625,33.833389,-91.158293,0
468148,2019-07-25 15:50:35,60495593109,"Kilback, Nitzsche and Leffler",travel,6.58,Randall,Dillon,M,4440 George Mills Suite 591,Dallas,...,32.7699,-96.7430,1263321,Television camera operator,1942-11-24,9ce4d94b27e36cc2c79bb02351831722,1343231435,32.458643,-96.577001,0


In [ ]:
# Define Features (X) and Target (y)
X = df_clean.drop(columns=["is_fraud"])
y = df_clean["is_fraud"]

## Test/Train Split


In [ ]:
# Stratification ensures identical fraud ratios in train and test splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y
)

In [8]:
# Feature Scaling for SVC distance calculations
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
# Calculate ratio for XGBoost class weighting
ratio = (len(y_train) - sum(y_train)) / sum(y_train)

## Model Training

In [ ]:
# SVC requires scaled features and stratified sub-sampling if dataset is massive
print("--- Training Model 1: Support Vector Classifier (SVC) ---")
svc_model = SVC(
    kernel="rbf",
    C=1.0,
    class_weight="balanced",
    # Enables probability outputs for ROC-AUC scoring & Streamlit confidence meters
    probability=True,
)

# For standard execution, fit on scaled training set:
svc_model.fit(X_train_scaled, y_train)

--- Training Model 1: Support Vector Classifier (SVC) ---


,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


In [ ]:
print("--- Training Model 2: XGBoost Classifier ---")
xgb_model = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=ratio,
    learning_rate=0.1,
    max_depth=6,
    n_jobs=-1,
)
xgb_model.fit(X_train_scaled, y_train)

--- Training Model 2: XGBoost Classifier ---


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


## Evaluate Models

In [ ]:
# Function to evaluate each model
def evaluate_model(model, name, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Model": name,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    }

    print(f"---------- {name} Evaluation Report ----------")
    print(classification_report(y_test, y_pred, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return metrics

In [ ]:
svc_results = evaluate_model(
    svc_model, "Support Vector Classifier (SVC)", X_test_scaled, y_test
)
xgb_results = evaluate_model(xgb_model, "XGBoost", X_test_scaled, y_test)

# Display side-by-side comparison
comparison_df = pd.DataFrame([svc_results, xgb_results])
print("---------- MODEL COMPARISON SUMMARY ----------")
print(comparison_df.to_string(index=False))

---------- Support Vector Classifier (SVC) Evaluation Report ----------
              precision    recall  f1-score   support

           0     0.9974    0.9792    0.9882      9946
           1     0.1229    0.5370    0.2000        54

    accuracy                         0.9768     10000
   macro avg     0.5602    0.7581    0.5941     10000
weighted avg     0.9927    0.9768    0.9840     10000

Confusion Matrix:
[[9739  207]
 [  25   29]]
---------- XGBoost Evaluation Report ----------
              precision    recall  f1-score   support

           0     0.9990    0.9968    0.9979      9946
           1     0.5789    0.8148    0.6769        54

    accuracy                         0.9958     10000
   macro avg     0.7890    0.9058    0.8374     10000
weighted avg     0.9967    0.9958    0.9962     10000

Confusion Matrix:
[[9914   32]
 [  10   44]]
---------- MODEL COMPARISON SUMMARY ----------
                          Model  Precision   Recall  F1-Score  ROC-AUC
Support Vector Cla

In [ ]:
# Save artifacts for streamlit app

print("\nSaving trained models and preprocessing artifacts...")
joblib.dump(svc_model, "svc_model.pkl")
joblib.dump(xgb_model, "xgb_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(label_encoders, "label_encoders.pkl")
joblib.dump(X.columns.tolist(), "feature_names.pkl")


Saving trained models and preprocessing artifacts...


['feature_names.pkl']